In [1]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from matplotlib.backends.backend_pdf import PdfPages

In [2]:
#from bulk, quality controlled data
qc_2_df = pd.read_parquet("../3.preprocessing_features/qc_report/Round_2_data_qc_report.parquet")

qc_1_df = pd.read_parquet("../3.preprocessing_features/qc_report/Round_1_data_qc_report.parquet")
qc_1_df["Metadata_condition"] = "standard"

qc_3_df = pd.read_parquet("../3.preprocessing_features/qc_report/Round_3_data_qc_report.parquet")

qc_4_df = pd.read_parquet("../3.preprocessing_features/qc_report/Round_4_data_qc_report.parquet")
pearson_df = pd.read_parquet("../5.optimization/results/round_1-4_pearson_correlation.parquet")


for df in (qc_1_df, qc_2_df, qc_3_df, qc_4_df, pearson_df):
    df["Metadata_cell_line"] = df["Metadata_cell_line"].str.replace("-", "", regex=False)

qc_df = pd.concat([qc_1_df, qc_2_df, qc_3_df, qc_4_df], ignore_index=True)
qc_df = qc_df.rename(columns={"Metadata_Plate": "Metadata_plate"})

In [3]:
pearson_df.head()

,Metadata_cell_line,Metadata_seeding_density,Metadata_time_point,Metadata_condition,Metadata_plate,Metadata_round,Shuffled,pearsons_correlation
0,A673,1000,24,synthemax,BR00145438,Round_2_data,False,0.926894
1,A673,1000,24,synthemax,BR00145438,Round_2_data,True,-0.084084
2,A673,1000,48,synthemax,BR00145439,Round_2_data,False,0.825755
3,A673,1000,48,synthemax,BR00145439,Round_2_data,True,0.010689
4,A673,1000,72,synthemax,BR00145440,Round_2_data,False,0.899398


In [4]:
qc_df_sorted = qc_df.sort_values(by="Metadata_cell_line")
qc_df_sorted.head()

,Metadata_cell_line,Metadata_seeding_density,Metadata_time_point,Metadata_condition,Metadata_plate,total_nuclei_segmented,total_failed_qc,percentage_failing_cells
435,A673,12000,72,synthemax,BR00145440,1726,512,29.663963
225,A673,8000,24,standard,BR00143976,2495,830,33.266533
224,A673,4000,24,standard,BR00143976,1801,511,28.373126
844,A673,8000,72,synthemax,BR00147000,4465,832,18.633819
165,A673,4000,48,standard,BR00143978,2960,985,33.277027


In [5]:
pearson_df.columns = pearson_df.columns.str.strip()
qc_df.columns = qc_df.columns.str.strip()
print("Columns in pearson_df:", pearson_df.columns)
print("Columns in qc_df:", qc_df.columns)

# Check data types of the columns we want to merge on
print("Data types in pearson_df:\n", pearson_df.dtypes)
print("Data types in qc_df:\n", qc_df.dtypes)

Columns in pearson_df: Index(['Metadata_cell_line', 'Metadata_seeding_density', 'Metadata_time_point',
       'Metadata_condition', 'Metadata_plate', 'Metadata_round', 'Shuffled',
       'pearsons_correlation'],
      dtype='object')
Columns in qc_df: Index(['Metadata_cell_line', 'Metadata_seeding_density', 'Metadata_time_point',
       'Metadata_condition', 'Metadata_plate', 'total_nuclei_segmented',
       'total_failed_qc', 'percentage_failing_cells'],
      dtype='object')
Data types in pearson_df:
 Metadata_cell_line           object
Metadata_seeding_density      int64
Metadata_time_point           int64
Metadata_condition           object
Metadata_plate               object
Metadata_round               object
Shuffled                     object
pearsons_correlation        float64
dtype: object
Data types in qc_df:
 Metadata_cell_line           object
Metadata_seeding_density      int64
Metadata_time_point           int64
Metadata_condition           object
Metadata_plate         

In [6]:
print(qc_df["Metadata_cell_line"].unique())
print(pearson_df["Metadata_cell_line"].unique())

['KNS42' 'KPNYN' 'NB1' 'ONS76' 'SJSA1' 'SKNAS' 'U2OS' 'A673' 'CHP212'
 'DAOY' 'G292' 'G401' 'G402' 'IMR32' 'PA1' 'SHSY5Y' 'SKNMC' 'Saos2'
 'CHLA10' 'CHLA113' 'CHLA200' 'CHLA218' 'CHLA25' 'SKNDZ' 'CHLA186'
 'CHLA194' 'CHLA196' 'CHLA203' 'CHLA210' 'CHLA258' 'CHLA262' 'COGAR397hnb'
 'COGE352' 'COGEP280' 'TC71' 'COGR466h' 'RD' 'RH30' 'RH4' 'RH41' 'SJGBM2'
 'COGW408' 'D283' 'CF1500 (Cure MEC Line #2)' 'CHP134'
 'CHP212 (EMEM:F12  preferred)' 'CHP2121020' 'CHP212EV' 'D425'
 'SHSY5Y(EMEM:F12  preferred)' 'SKNAS1020' 'X0092 (Cure MEC Line #1)'
 'ATRT310' 'PBT05FHTC' 'ATRT311 FHTC' 'EPD210 FHTC' 'PBT04 FHTC'
 'CCD 841 CoN' 'HMC3' 'SKNASEV' 'WI38' 'WPMY1']
['A673' 'ATRT310' 'ATRT311 FHTC' 'CCD 841 CoN' 'CF1500 (Cure MEC Line #2)'
 'CHLA10' 'CHLA113' 'CHLA186' 'CHLA194' 'CHLA196' 'CHLA200' 'CHLA203'
 'CHLA210' 'CHLA218' 'CHLA25' 'CHLA258' 'CHLA262' 'CHP134' 'CHP212'
 'CHP212 (EMEM:F12  preferred)' 'CHP2121020' 'CHP212EV' 'COGAR397hnb'
 'COGE352' 'COGEP280' 'COGR466h' 'COGW408' 'D283' 'D425' 'DAOY

In [7]:
# Merge pearson_df and qc_df
merged_df = pd.merge(
    pearson_df[pearson_df["Shuffled"] == "False"],
    qc_df,
    on=["Metadata_cell_line", "Metadata_seeding_density", "Metadata_time_point", "Metadata_condition", "Metadata_plate"],
    how="inner"
)

# save df
merged_df.to_parquet("../5.optimization/results/merged_pearson_qc_data.parquet")
print("Merged dataframe saved to results/merged_pearson_qc_data.parquet")

Merged dataframe saved to results/merged_pearson_qc_data.parquet


In [8]:
merged_df.head()

,Metadata_cell_line,Metadata_seeding_density,Metadata_time_point,Metadata_condition,Metadata_plate,Metadata_round,Shuffled,pearsons_correlation,total_nuclei_segmented,total_failed_qc,percentage_failing_cells
0,A673,1000,24,synthemax,BR00145438,Round_2_data,False,0.926894,1338,145,10.837070
1,A673,1000,48,synthemax,BR00145439,Round_2_data,False,0.825755,1772,299,16.873589
2,A673,1000,72,synthemax,BR00145440,Round_2_data,False,0.899398,4091,417,10.193107
3,A673,2000,24,synthemax,BR00145438,Round_2_data,False,0.930421,2178,535,24.563820
4,A673,2000,48,synthemax,BR00145439,Round_2_data,False,0.894138,3388,568,16.765053


In [9]:
custom_palette = sns.color_palette("Set1", n_colors=5)
# Create a PdfPages object to save all plots in a single PDF
with PdfPages("../5.optimization/results/rounds_1-4_pearson_vs_percentage_failing_cells.pdf") as pdf:
    # Loop over each cell line
    for cell_line in merged_df["Metadata_cell_line"].unique():
        cell_line_df = merged_df[merged_df["Metadata_cell_line"] == cell_line]

        # One figure with a column‑panel for every condition
        g = sns.relplot(
            data=cell_line_df,
            x="pearsons_correlation",
            y="percentage_failing_cells",
            hue="Metadata_seeding_density",
            style="Metadata_time_point",
            markers=["o", "X", "s"],      # list length ≥ number of unique time points
            palette=custom_palette,
            kind="scatter",
            col="Metadata_condition",      # ← facet by condition
            col_wrap=None,                 # all panels in a single row; use an int to wrap
            height=6,
            aspect=1,
        )

        g.axes.flat[0].invert_yaxis() 

        # Overall title & axis labels
        g.fig.suptitle(
            f"Pearson Correlation vs Percentage Failing Cells\nCell Line: {cell_line}",
            fontsize=16,
            y=1.1  # move title a bit up so it doesn’t overlap
        )
        g.set_axis_labels("Pearson Correlation", "Percentage Failing Cells")


        # Move legend outside the grid
        g._legend.set_title("Seeding Density")
        g._legend.set_bbox_to_anchor((1, 1))
        g._legend.set_loc("upper left")

        # Save and close
        pdf.savefig(g.fig, bbox_inches="tight", transparent=True)
        plt.close(g.fig)

print("Plots saved to results/round_1-4_pearson_vs_percentage_failing_cells.pdf")

/tmp/ipykernel_3860042/3903914713.py:9: UserWarning: The palette list has more values (5) than needed (4), which may not be intended.
  g = sns.relplot(


Plots saved to results/round_1-4_pearson_vs_percentage_failing_cells.pdf
